# Ev Fiyatlarını Makine Öğrenmesi ile Tahmin Etmek: Uçtan Uca Bir Regresyon Projesi
Türkiye Yapay Zeka Akademisi — Uçtan Uca Makine Öğrenmesi Projesi Final Ödevi

In [ ]:
"""
PROJE: Ev Fiyatlarını Makine Öğrenmesi ile Tahmin Etmek (Uçtan Uca Regresyon Projesi)

AMAÇ:
    Kaliforniya'daki mahallelere ait demografik ve coğrafi özelliklerden yola
    çıkarak bölgedeki evlerin medyan fiyatını (median_house_value) tahmin
    etmek. Problem türü: REGRESYON.

KULLANILAN KÜTÜPHANELER:
    - pandas, numpy      : veri okuma / işleme
    - matplotlib, seaborn : görselleştirme
    - scikit-learn        : ön işleme, modelleme, değerlendirme

ÇALIŞTIRMA ADIMLARI:
    1. Bu notebook'taki tüm hücreleri sırasıyla (yukarıdan aşağıya) çalıştırın.
    2. İnternet bağlantısı gereklidir; veri seti kod içinde otomatik olarak
       aşağıdaki adresten indirilir, ekstra bir dosya indirmenize gerek yoktur:
       https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv
    3. Notebook baştan sona hatasız çalışacak şekilde hazırlanmıştır, hiçbir
       hücrede elle tamamlama (TODO) gerekmez.
"""
print("Proje docstring'i yukarıda tanımlanmıştır. Notebook çalıştırılmaya hazır.")


## 2. Kütüphanelerin Import Edilmesi

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

print("Kütüphaneler başarıyla yüklendi.")


## 3. Veri Setinin Yüklenmesi
**Veri seti:** California Housing Prices (1990 ABD nüfus sayımı verilerinden türetilmiştir).

**Kaynak:** Veri seti, Aurélien Géron'un *Hands-On Machine Learning* kitabının resmi GitHub deposunda barındırılmaktadır ve `pandas.read_csv` ile doğrudan bir URL üzerinden otomatik olarak indirilir; ek bir dosya indirip proje klasörüne koymanıza gerek yoktur:
`https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv`

**Neyi temsil ediyor:** Her satır, Kaliforniya'daki bir mahalleyi (census block group) temsil eder. Sütunlar konum, demografi (nüfus, hane sayısı, medyan gelir) ve konut özellikleri (oda sayısı, yaş vb.) içerir.

**Hedef değişken:** Veri setinde hazır bir `price` sütunu bulunmamaktadır; bunun yerine gerçek hedef sütun **`median_house_value`** olup mahalledeki evlerin medyan (ortanca) satış fiyatını (USD) temsil eder. Bu proje boyunca hedef değişken olarak `median_house_value` kullanılacaktır.

**Satır sayısı:** 20.640 satır (ödevde istenen en az 200 satır şartını fazlasıyla karşılamaktadır).

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv"
TARGET_COL = "median_house_value"

df = pd.read_csv(DATA_URL)

print(f"Veri seti başarıyla indirildi. Boyut: {df.shape[0]} satır, {df.shape[1]} sütun.")
df.head()


## 4. Problem Türü

In [ ]:
print("PROBLEM TÜRÜ: REGRESYON")
print(f"HEDEF DEĞİŞKEN: '{TARGET_COL}' (mahalledeki evlerin medyan satış fiyatı, USD)")
print(f"Hedef değişken veri tipi: {df[TARGET_COL].dtype}")
print(f"Hedef değişken istatistikleri:\n{df[TARGET_COL].describe()}")


## 5. Temel Veri İncelemesi

In [ ]:
print("=== head() ===")
display(df.head())

print("\n=== shape ===")
print(df.shape)

print("\n=== info() ===")
df.info()

print("\n=== describe() ===")
display(df.describe())


In [ ]:
# Kategorik ve sayısal sütunların belirlenmesi
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()

print(f"Kategorik sütunlar ({len(categorical_cols)}): {categorical_cols}")
print(f"Sayısal sütunlar ({len(numerical_cols)}): {numerical_cols}")

print("\nKategorik sütunun benzersiz değerleri:")
for col in categorical_cols:
    print(f"  {col}: {df[col].unique()}")


## 6. Eksik Değer Analizi

In [ ]:
missing_counts = df.isnull().sum()
missing_cols = missing_counts[missing_counts > 0]

if len(missing_cols) > 0:
    print("Eksik değer bulunan sütunlar:")
    missing_summary = pd.DataFrame({
        "Eksik Değer Sayısı": missing_cols,
        "Oran (%)": (missing_cols / len(df) * 100).round(2)
    })
    display(missing_summary)
else:
    print("Veri setinde eksik değer bulunmamaktadır.")


## 7. Veri Temizleme (Eksik Değerlerin Doldurulması)
`total_bedrooms` sütunundaki eksik değerler, aykırı değerlerden fazla etkilenmemesi için medyan ile doldurulmuştur. Bu, dersin kapsamındaki basit ve yaygın kullanılan bir yaklaşımdır. Not: Daha titiz bir pipeline'da bu doldurma işlemi yalnızca eğitim (train) verisi üzerinden öğrenilip test verisine uygulanır; burada ödevin istediği notebook sırası (temizleme adımı, split adımından önce) izlenmiştir. Tek bir sayısal sütunun medyanla doldurulması hedef değişkeni içermediğinden anlamlı bir veri sızıntısına yol açmaz.

In [ ]:
bedrooms_median = df["total_bedrooms"].median()
df["total_bedrooms"] = df["total_bedrooms"].fillna(bedrooms_median)

print(f"'total_bedrooms' sütunundaki eksik değerler medyan ({bedrooms_median:.2f}) ile dolduruldu.")
print(f"Doldurma sonrası toplam eksik değer sayısı: {df.isnull().sum().sum()}")


## 8. Aykırı Değer Analizi

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
outlier_check_cols = ["housing_median_age", "total_rooms", "total_bedrooms",
                       "population", "households", "median_income", TARGET_COL]

for ax, col in zip(axes.ravel(), outlier_check_cols):
    sns.boxplot(y=df[col], ax=ax, color="#2F6F52")
    ax.set_title(col)

axes.ravel()[-1].axis("off")
plt.suptitle("Sayısal Değişkenlerde Aykırı Değer İncelemesi (Boxplot)", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# IQR yöntemiyle aykırı değer oranlarının incelenmesi (bilgi amaçlı)
def iqr_outlier_ratio(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outliers = series[(series < lower) | (series > upper)]
    return len(outliers), round(len(outliers) / len(series) * 100, 2)

print("IQR yöntemine göre aykırı değer oranları:")
for col in outlier_check_cols:
    count, pct = iqr_outlier_ratio(df[col])
    print(f"  {col}: {count} adet (%{pct})")

# YORUM: Aykırı değerler otomatik olarak SİLİNMEMİŞTİR. Örneğin median_income ve
# total_rooms gibi sütunlardaki yüksek değerler gerçek dünyada var olan, anlamlı
# mahalleleri temsil edebilir (örn. çok yüksek gelirli ya da çok kalabalık bölgeler).
# Bu satırları silmek, modelin öğrenebileceği gerçek örüntüleri kaybetmesine yol
# açabilir. Bunun yerine, sadece hedef değişkende (median_house_value) veri
# toplama sürecinden kaynaklanan bir üst sınır (cap) sorunu olup olmadığı kontrol edilir.
capped_count = (df[TARGET_COL] == df[TARGET_COL].max()).sum()
print(f"\nHedef değişkenin maksimum değerine ({df[TARGET_COL].max()}) eşit satır sayısı: {capped_count}")
print("Bu satırlar, veri toplama sırasında oluşmuş bir üst sınır (cap) etkisi taşıyabileceğinden çıkarılmıştır.")

df = df[df[TARGET_COL] < df[TARGET_COL].max()].reset_index(drop=True)
print(f"Temizleme sonrası veri seti boyutu: {df.shape}")


## 9. Feature Engineering (En Az 2 Yeni Öznitelik)
Veri setinin gerçek sütunları (`total_rooms`, `total_bedrooms`, `population`, `households`) incelenerek aşağıdaki iki yeni ve anlamlı öznitelik oluşturulmuştur:

- **rooms_per_household**: Bir hanedeki ortalama oda sayısı (`total_rooms / households`). Mahallenin konut büyüklüğü/refah seviyesi hakkında `total_rooms` tek başına verdiğinden daha anlamlı bilgi verir.
- **bedrooms_per_room**: Toplam odalar içindeki yatak odası oranı (`total_bedrooms / total_rooms`). Düşük oran genellikle daha büyük/lüks evleri işaret eder ve fiyatla ilişkilidir.
- **population_per_household**: Bir hanedeki ortalama kişi sayısı (`population / households`). Hane yoğunluğunu ölçer, kalabalık/aile büyüklüğü ile fiyat arasındaki ilişkiyi yakalamaya yardımcı olur.

In [ ]:
df["rooms_per_household"] = df["total_rooms"] / df["households"]
df["bedrooms_per_room"] = df["total_bedrooms"] / df["total_rooms"]
df["population_per_household"] = df["population"] / df["households"]

new_features = ["rooms_per_household", "bedrooms_per_room", "population_per_household"]
print("Oluşturulan yeni öznitelikler:")
display(df[new_features].describe())


## 10. Kategorik Değişkenlerin Encoding İşlemi
`ocean_proximity` kategorik sütunu `OneHotEncoder` ile sayısal forma dönüştürülmüştür. Encoder, kategori bilgisini (sabit ve sınırlı sayıda değer alan bir sütunu) öğrendiğinden ve hedef değişkeni kullanmadığından, split öncesinde uygulanması anlamlı bir veri sızıntısına yol açmaz.

In [ ]:
ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
ocean_encoded = ohe.fit_transform(df[["ocean_proximity"]])
ocean_encoded_cols = ohe.get_feature_names_out(["ocean_proximity"])

ocean_df = pd.DataFrame(ocean_encoded, columns=ocean_encoded_cols, index=df.index)
df_encoded = pd.concat([df.drop(columns=["ocean_proximity"]), ocean_df], axis=1)

print("Encoding sonrası oluşan yeni sütunlar:", list(ocean_encoded_cols))
df_encoded.head()


## 11. Train / Validation / Test Ayrımı
**Veri sızıntısını (data leakage) önlemek için** train/validation/test ayrımı, korelasyon tabanlı feature selection adımından ÖNCE yapılmıştır: bu sayede feature seçimi yalnızca training verisine bakılarak yapılacak, validation ve test verisindeki hiçbir bilgi seçim kararını etkilemeyecektir.

Veri seti %70 training, %15 validation, %15 test olacak şekilde ayrılmıştır. `random_state=42` ile tekrarlanabilirlik sağlanmıştır. Test verisi, notebook boyunca yalnızca final değerlendirme (Bölüm 18) için kullanılacaktır.

In [ ]:
# Ham (henüz feature selection uygulanmamış) X ve y
X_full = df_encoded.drop(columns=[TARGET_COL])
y_full = df_encoded[TARGET_COL]

X_train_full, X_temp_full, y_train, y_temp = train_test_split(
    X_full, y_full, test_size=0.30, random_state=RANDOM_STATE
)
X_val_full, X_test_full, y_val, y_test = train_test_split(
    X_temp_full, y_temp, test_size=0.50, random_state=RANDOM_STATE
)

print(f"Training set:   {X_train_full.shape[0]} satır (%{X_train_full.shape[0]/len(X_full)*100:.1f})")
print(f"Validation set: {X_val_full.shape[0]} satır (%{X_val_full.shape[0]/len(X_full)*100:.1f})")
print(f"Test set:       {X_test_full.shape[0]} satır (%{X_test_full.shape[0]/len(X_full)*100:.1f})")


## 12. Feature Selection (Öznitelik Seçimi)
Öznitelik seçimi için **korelasyon analizi** kullanılmıştır. Veri sızıntısını önlemek için korelasyon katsayıları **yalnızca training verisi (`X_train_full`, `y_train`) üzerinden** hesaplanmış, validation ve test verisi bu hesaplamaya hiç dahil edilmemiştir. Her özniteliğin hedef değişkenle Pearson korelasyon katsayısının mutlak değeri hesaplanmış, |korelasyon| < 0.05 olan (hedefle neredeyse hiç ilişkisi olmayan) öznitelikler elenmiştir. Seçilen sütunlar daha sonra validation ve test setlerine de (yeniden hesaplama yapılmadan) aynen uygulanmıştır.

In [ ]:
train_with_target = X_train_full.copy()
train_with_target[TARGET_COL] = y_train

corr_with_target = train_with_target.corr(numeric_only=True)[TARGET_COL].drop(TARGET_COL)
corr_sorted = corr_with_target.abs().sort_values(ascending=False)

print("Özniteliklerin hedef değişkenle korelasyonu — SADECE training verisi üzerinden (mutlak değer, büyükten küçüğe):")
display(corr_sorted)

CORR_THRESHOLD = 0.05
selected_features = corr_sorted[corr_sorted >= CORR_THRESHOLD].index.tolist()
dropped_features = corr_sorted[corr_sorted < CORR_THRESHOLD].index.tolist()

print(f"\nSeçilen öznitelikler ({len(selected_features)}): {selected_features}")
print(f"Elenen öznitelikler ({len(dropped_features)}): {dropped_features}")

plt.figure(figsize=(10, 8))
sns.heatmap(train_with_target[selected_features + [TARGET_COL]].corr(), annot=True, fmt=".2f",
            cmap="RdYlGn", center=0)
plt.title("Seçilen Öznitelikler için Korelasyon Heatmap (Training Verisi)")
plt.tight_layout()
plt.show()

# Seçilen öznitelikler, hesaplama yapılmadan train/val/test setlerine uygulanır
X_train = X_train_full[selected_features]
X_val = X_val_full[selected_features]
X_test = X_test_full[selected_features]

print(f"\nModelleme için nihai öznitelik matrisi boyutu (train): {X_train.shape}")


## 13. Ölçekleme (Scaling)
Linear Regression ve Ridge Regression gibi mesafe/katsayı tabanlı modeller için `StandardScaler` uygulanmıştır. **Random Forest gibi tree-based (ağaç tabanlı) modeller için ölçekleme gerekli değildir**, çünkü bu modeller özniteliklerin mutlak büyüklüğüne değil, bölme (split) noktalarına dayanır. Veri sızıntısını önlemek için `scaler` **yalnızca training verisi üzerinde `fit` edilmiş**, ardından validation ve test verilerine sadece `transform` uygulanmıştır.

In [ ]:
scaler = StandardScaler()
scaler.fit(X_train)  # yalnızca training verisi üzerinde fit edilir (leakage önlenir)

X_train_scaled = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns, index=X_train.index)
X_val_scaled = pd.DataFrame(scaler.transform(X_val), columns=X_val.columns, index=X_val.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

print("Ölçekleme tamamlandı. (Linear Regression ve Ridge için ölçekli veri; Random Forest için ölçeksiz veri kullanılacaktır.)")
X_train_scaled.head()


## 14. Model Eğitimi (En Az 3 Model)
Üç farklı regresyon modeli eğitilmiştir: **Linear Regression**, **Ridge Regression** ve **Random Forest Regressor**.

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0, random_state=RANDOM_STATE),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
}

# Doğrusal modeller ölçekli veri, Random Forest ölçeksiz (orijinal) veri ile eğitilir
uses_scaled_data = {"Linear Regression": True, "Ridge": True, "Random Forest": False}

trained_models = {}
for name, model in models.items():
    X_tr = X_train_scaled if uses_scaled_data[name] else X_train
    model.fit(X_tr, y_train)
    trained_models[name] = model
    print(f"{name} eğitildi.")


## 15. Validation Setinde Model Karşılaştırması

In [ ]:
def evaluate(model, X_eval, y_true):
    preds = model.predict(X_eval)
    mae = mean_absolute_error(y_true, preds)
    mse = mean_squared_error(y_true, preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, preds)
    return mae, mse, rmse, r2

val_results = []
for name, model in trained_models.items():
    X_ev = X_val_scaled if uses_scaled_data[name] else X_val
    mae, mse, rmse, r2 = evaluate(model, X_ev, y_val)
    val_results.append({"Model": name, "MAE": mae, "MSE": mse, "RMSE": rmse, "R2": r2})

val_results_df = pd.DataFrame(val_results).set_index("Model")
print("=== Validation Model Karşılaştırması ===")
display(val_results_df.round(2))

best_model_name = val_results_df["R2"].idxmax()
print(f"\nEn başarılı model (validation R² skoruna göre): {best_model_name}")


## 16. Cross-Validation

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print("=== 5-Fold Cross-Validation Sonuçları (RMSE) ===")
cv_results = {}
for name, model in models.items():
    X_cv = X_train_scaled if uses_scaled_data[name] else X_train
    neg_mse_scores = cross_val_score(model, X_cv, y_train, cv=kf,
                                      scoring="neg_mean_squared_error", n_jobs=-1)
    rmse_scores = np.sqrt(-neg_mse_scores)
    cv_results[name] = (rmse_scores.mean(), rmse_scores.std())
    print(f"{name}: Ortalama RMSE = {rmse_scores.mean():.2f}  |  Std = {rmse_scores.std():.2f}")


## 17. Hyperparameter Tuning
En başarılı model için `GridSearchCV` kullanılarak makul boyutlu bir hiperparametre araması yapılmıştır.

In [ ]:
param_grids = {
    "Ridge": {"alpha": [0.01, 0.1, 1.0, 10.0, 50.0, 100.0]},
    "Random Forest": {
        "n_estimators": [100],
        "max_depth": [10, None],
        "min_samples_leaf": [1, 2],
    },
}

X_train_for_best = X_train_scaled if uses_scaled_data[best_model_name] else X_train

if best_model_name == "Linear Regression":
    print("Linear Regression'ın ayarlanacak bir hiperparametresi bulunmamaktadır "
          "(kapalı-form çözüm kullanır). Bu nedenle GridSearchCV atlanmış, model "
          "olduğu gibi final model olarak kullanılacaktır.")
    best_model = trained_models["Linear Regression"]
    best_params = {}
    best_cv_score = cv_results["Linear Regression"][0]
else:
    base_model = models[best_model_name]
    grid = GridSearchCV(
        base_model,
        param_grid=param_grids[best_model_name],
        cv=5,
        scoring="neg_mean_squared_error",
        n_jobs=-1,
    )
    grid.fit(X_train_for_best, y_train)
    best_model = grid.best_estimator_
    best_params = grid.best_params_
    best_cv_score = np.sqrt(-grid.best_score_)

print(f"\nEn iyi model: {best_model_name}")
print(f"En iyi parametreler (best_params_): {best_params}")
print(f"En iyi CV RMSE skoru (best_score_'dan hesaplanan): {best_cv_score:.2f}")


## 18. Final Test Değerlendirmesi
Hyperparameter tuning sonrası elde edilen en iyi model, notebook boyunca hiç kullanılmamış olan **test verisi** üzerinde değerlendirilmiştir.

In [ ]:
X_test_for_best = X_test_scaled if uses_scaled_data[best_model_name] else X_test

test_mae, test_mse, test_rmse, test_r2 = evaluate(best_model, X_test_for_best, y_test)

print("=== Final Test Sonuçları ===")
print(f"MAE  : {test_mae:.2f}")
print(f"MSE  : {test_mse:.2f}")
print(f"RMSE : {test_rmse:.2f}")
print(f"R2   : {test_r2:.4f}")

test_predictions = best_model.predict(X_test_for_best)
comparison_df = pd.DataFrame({
    "Gerçek Fiyat": y_test.values,
    "Tahmin Edilen Fiyat": test_predictions
}).reset_index(drop=True)

print("\nGerçek ve tahmin edilen fiyatlardan örnek satırlar:")
display(comparison_df.head(10))


In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(comparison_df["Gerçek Fiyat"], comparison_df["Tahmin Edilen Fiyat"],
            alpha=0.3, color="#2F6F52", s=15)
min_val = min(comparison_df["Gerçek Fiyat"].min(), comparison_df["Tahmin Edilen Fiyat"].min())
max_val = max(comparison_df["Gerçek Fiyat"].max(), comparison_df["Tahmin Edilen Fiyat"].max())
plt.plot([min_val, max_val], [min_val, max_val], "r--", linewidth=2, label="İdeal Tahmin (y=x)")
plt.xlabel("Gerçek Fiyat (median_house_value)")
plt.ylabel("Tahmin Edilen Fiyat")
plt.title(f"Gerçek vs Tahmin Edilen Fiyat — {best_model_name} (Test Seti)")
plt.legend()
plt.tight_layout()
plt.show()


## 19. Feature Importance / Explainability

In [ ]:
if hasattr(best_model, "feature_importances_"):
    importance_df = pd.DataFrame({
        "Öznitelik": X_train_for_best.columns,
        "Önem Skoru": best_model.feature_importances_
    }).sort_values("Önem Skoru", ascending=False).reset_index(drop=True)

    print(f"{best_model_name} - Feature Importance:")
    display(importance_df)

    plt.figure(figsize=(9, 6))
    sns.barplot(data=importance_df, x="Önem Skoru", y="Öznitelik", color="#2F6F52")
    plt.title(f"{best_model_name} - Feature Importance")
    plt.tight_layout()
    plt.show()

elif hasattr(best_model, "coef_"):
    coef_df = pd.DataFrame({
        "Öznitelik": X_train_for_best.columns,
        "Katsayı": best_model.coef_
    })
    coef_df["Mutlak Katsayı"] = coef_df["Katsayı"].abs()
    coef_df = coef_df.sort_values("Mutlak Katsayı", ascending=False).reset_index(drop=True)

    print(f"{best_model_name} - Katsayılar (Coefficients):")
    display(coef_df)

    plt.figure(figsize=(9, 6))
    sns.barplot(data=coef_df, x="Katsayı", y="Öznitelik", color="#2F6F52")
    plt.title(f"{best_model_name} - Katsayı Önemleri")
    plt.tight_layout()
    plt.show()


## 20. Sonuçların Özeti

In [ ]:
print("=" * 60)
print("VALIDATION MODEL COMPARISON")
print("=" * 60)
display(val_results_df[["MAE", "RMSE", "R2"]].round(2))

print("\n" + "=" * 60)
print("FINAL MODEL")
print("=" * 60)
print(f"Best Model      : {best_model_name}")
print(f"Best Parameters : {best_params}")
print(f"Test MAE        : {test_mae:.2f}")
print(f"Test MSE        : {test_mse:.2f}")
print(f"Test RMSE       : {test_rmse:.2f}")
print(f"Test R2         : {test_r2:.4f}")


## 21. Model Yorumu ve Sonuç Değerlendirmesi
*(Bu bölümdeki yorumlar, yukarıdaki hücrelerin gerçek çalıştırma çıktılarına göre yeniden değerlendirilmelidir; genel çerçeve aşağıdaki gibidir.)*

- **Hangi model daha başarılı oldu?** Validation karşılaştırma tablosunda en yüksek R² ve en düşük hata (MAE/RMSE) değerlerine sahip model, `best_model_name` değişkeninde tutulan modeldir (genellikle Random Forest, doğrusal olmayan ilişkileri yakalayabildiği için Linear Regression ve Ridge'den daha iyi performans gösterir).
- **Neden tercih edildi?** Hem validation setindeki metrikler hem de cross-validation sonuçları (ortalama RMSE ve düşük standart sapma) tutarlı biçimde bu modeli işaret etmektedir; ayrıca hiperparametre araması sonrası test performansı da bunu doğrulamaktadır.
- **Hangi özellikler daha önemli?** Bölüm 19'daki feature importance / katsayı tablosunda genellikle `median_income` (medyan gelir) ve konum bilgisi (`latitude`, `longitude`) en yüksek öneme sahip özniteliklerdir; bu, gelir düzeyinin ve coğrafi konumun ev fiyatlarını belirlemede kritik rol oynadığını göstermektedir.
- **Modelin güçlü yönleri:** Random Forest, doğrusal olmayan ilişkileri ve öznitelikler arası etkileşimleri yakalayabilir, aykırı değerlere Linear Regression'a kıyasla daha dayanıklıdır.
- **Modelin sınırlılıkları:** Eğitim verisinin kapsamadığı bölgelerde (örn. çok uç gelir/konum kombinasyonları) tahmin performansı düşebilir; model, veri setinin toplandığı 1990 yılı Kaliforniya piyasa koşullarını yansıtır, güncel piyasaya doğrudan uygulanamaz.
- **Gerçek hayatta nerede hata yapabilir?** Piyasa koşulları zamanla değiştiğinden (enflasyon, faiz oranları, bölgesel gelişim) model zamanla güncelliğini yitirebilir; ayrıca eğitim verisinde az temsil edilen mahalle tiplerinde (örn. çok yüksek fiyatlı lüks bölgeler) tahminler daha az güvenilir olabilir.